## E0.1 CTE basica - Estaditicas por zona

In [0]:
%sql

WITH 
 estadisticas_por_zona AS (
   SELECT
    pickup_zip AS zona,
    COUNT(*) AS total_viajes,
    ROUND(AVG(trip_distance),2) AS distancia_promedio,
    ROUND(AVG(fare_amount),2) AS tarifa_promedio
   FROM samples.nyctaxi.trips
   WHERE(
      pickup_zip IS NOT NULL AND
      trip_distance > 0 AND
      fare_amount > 0
   )
   GROUP BY pickup_zip
   HAVING total_viajes > 100
 )

 SELECT * FROM estadisticas_por_zona
 ORDER BY zona;
 


## E0.2 - CTE multiple - Analisis comparativo

In [0]:
%sql

WITH
    promedio_hora AS (
        SELECT
            HOUR(tpep_pickup_datetime) AS hora,
            ROUND(AVG(fare_amount),2) AS tarifa_promedio,
            COUNT(*) AS total_viajes,
            ROUND(AVG(trip_distance),2) AS distancia_promedio
        FROM samples.nyctaxi.trips
        WHERE fare_amount>0
        GROUP BY hora
    ),
    promedio_gral AS (
        SELECT
            ROUND(AVG(fare_amount),2) as tarifa_promedio_gral
        FROM samples.nyctaxi.trips
        WHERE fare_amount>0

    )


    SELECT
        ph.hora,
        ph.total_viajes,
        ph.distancia_promedio,
        ph.tarifa_promedio,
        pg.tarifa_promedio_gral,
        ROUND(ph.tarifa_promedio - pg.tarifa_promedio_gral,2) AS diferencia_promedios,
        ROUND((diferencia_promedios*100.0)/pg.tarifa_promedio_gral,2) AS porcentaje_diferencia
    FROM promedio_hora AS ph
    CROSS JOIN promedio_gral AS pg
    ORDER BY hora
    

## E0.3 CTE para limpieza de datos

In [0]:
%sql

WITH 
    viajes_validos AS (
        SELECT *
        FROM samples.nyctaxi.trips
        WHERE
            trip_distance > 0 AND
            fare_amount > 0        
    )


SELECT
    ROUND(MIN(trip_distance),2) AS `DISTANCIA MINIMA`,
    ROUND(MAX(trip_distance),2) AS `DISTANCIA MAXIMA`,
    ROUND(AVG(trip_distance),2) AS `DISTANCIA PROMEDIO`,
    ROUND(MEDIAN(trip_distance),2) AS `DISTANCIA MEDIA`
FROM viajes_validos




## E0.4 - Window Function - ROW_NUMBER ranking

In [0]:
%sql

select
    tpep_pickup_datetime as fecha,
    trip_distance as distancia,
    fare_amount as tarifa,
    row_number() over (order by fare_amount desc) as ranking
from samples.nyctaxi.trips
where trip_distance > 0 and fare_amount > 0
limit 20



## E0.5 - Window Function -RANK, DENSE_RANK, ROW_NUMBER

In [0]:
%sql

WITH 
 cantidad_por_zona as(
    SELECT
    COUNT(*) as cantidad,
    pickup_zip as zona
    from samples.nyctaxi.trips
    GROUP by pickup_zip    
 )

-- cantidad por zonas orden asc
SELECT *
FROM cantidad_por_zona;



In [0]:
%sql

WITH 
 cantidad_por_zona as(
    SELECT
    COUNT(*) as cantidad,
    pickup_zip as zona
    from samples.nyctaxi.trips
    GROUP by pickup_zip    
 )
 -- ranking row_number
 SELECT 
 row_number() over (order by cantidad desc) as ranking,
 zona,
 cantidad
 from cantidad_por_zona
 limit 10


In [0]:
%sql

WITH 
 cantidad_por_zona as(
    SELECT
    COUNT(*) as cantidad,
    pickup_zip as zona
    from samples.nyctaxi.trips
    GROUP by pickup_zip    
 )
 -- ranking rank
 SELECT 
 rank() over (order by cantidad desc) as ranking,
 zona,
 cantidad
 from cantidad_por_zona
 limit 10


In [0]:
%sql

WITH 
 cantidad_por_zona as(
    SELECT
    COUNT(*) as cantidad,
    pickup_zip as zona
    from samples.nyctaxi.trips
    GROUP by pickup_zip    
 )
 -- ranking dense_rank
 SELECT 
 dense_rank() over (order by cantidad desc) as ranking,
 zona,
 cantidad
 from cantidad_por_zona
 limit 10


In [0]:
%sql

WITH 
 cantidad_por_zona as(
    SELECT
    COUNT(*) as cantidad,
    pickup_zip as zona
    from samples.nyctaxi.trips
    where pickup_zip is not null
    GROUP by pickup_zip
    ORDER BY cantidad desc
    limit 10
 )
 -- ranking row_number, rank, dense_rank
 SELECT
 zona,
 cantidad,
 row_number() over (order by cantidad desc) as row_number,
 rank() over (order by cantidad desc) as rank,
 dense_rank() over (order by cantidad desc) as dense 
 from cantidad_por_zona
 ORDER BY cantidad desc;

-- WITH viajes_por_zona AS (
--   SELECT 
--     pickup_zip,
--     COUNT(*) AS cantidad_viajes
--   FROM samples.nyctaxi.trips
--   WHERE pickup_zip IS NOT NULL
--   GROUP BY pickup_zip
--   ORDER BY cantidad_viajes DESC
--   LIMIT 10
-- )
-- SELECT 
--   pickup_zip,
--   cantidad_viajes,
--   ROW_NUMBER() OVER (ORDER BY cantidad_viajes DESC) AS ranking_row_number,
--   RANK()       OVER (ORDER BY cantidad_viajes DESC) AS ranking_rank,
--   DENSE_RANK() OVER (ORDER BY cantidad_viajes DESC) AS ranking_dense_rank
-- FROM viajes_por_zona
-- ORDER BY cantidad_viajes DESC;
 


## E0.6 - Comparar con promedio gral

In [0]:
%sql

select
    tpep_pickup_datetime as fecha,
    row_number() over (order by tpep_pickup_datetime)as `id`,
    fare_amount as tarifa,
    trip_distance as distancia,
    ROUND(AVG(fare_amount) over(), 4) as t_prom_gral,
    ROUND(tarifa - t_prom_gral, 4) as dif_t_tpg,
    ROUND((dif_t_tpg*100.0)/t_prom_gral, 4) as `%_dif`    
    from samples.nyctaxi.trips
    where fare_amount>0
    order by fecha desc
    limit 20;

    -- es recomendable repetir los calculos en cada operacion necesaria dado que si se utilizar los campos calculados se pierden datos por el redondeo

In [0]:
%sql
SELECT 
  tpep_pickup_datetime AS fecha,
  row_number() OVER (ORDER BY tpep_pickup_datetime) AS id,
  ROUND(fare_amount, 2) AS tarifa,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(AVG(fare_amount) OVER(), 2) AS tarifa_promedio_general,
  ROUND(fare_amount - AVG(fare_amount) OVER(), 2) AS diferencia_con_promedio,
  ROUND((fare_amount - AVG(fare_amount) OVER()) * 100.0 / AVG(fare_amount) OVER(), 2) AS porcentaje_diferencia
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fecha DESC
LIMIT 20;

## E0.7 - Promedio por particion

In [0]:
%sql

select
    tpep_pickup_datetime as fecha,
    round(fare_amount,4) as tarifa,
    pickup_zip as `zona de inicio`,
    round(avg(fare_amount) over (partition by pickup_zip),4) as `tarifa promedio por zona`
    from samples.nyctaxi.trips
    order by pickup_zip
    limit 20;


## E0.8 - LAG() para comparar con anterior

In [0]:
%sql

select
    tpep_pickup_datetime as fecha,
    round(fare_amount, 4) as tarifa,
    coalesce(lag(fare_amount) over (order by tpep_pickup_datetime), -1.0) as tarifa_anterior,
    if(tarifa_anterior>=0, tarifa-tarifa_anterior,null) as diferencia
    from samples.nyctaxi.trips
    order by tpep_pickup_datetime
    limit 20;

## E0.9 - SUM acumulado

In [0]:
%sql

select
    tpep_pickup_datetime as fecha,
    round(fare_amount, 4) as tarifa,
    round(sum(fare_amount) over (order by tpep_pickup_datetime rows between unbounded preceding and current row),4) as acumulado
    from samples.nyctaxi.trips
    order by tpep_pickup_datetime;


## E0.10 - Combinando CTEs y Window functions

In [0]:
%sql

with viajes_validos as (
    select *
    from samples.nyctaxi.trips
    where fare_amount>0 and trip_distance>0
),
promedios_por_zona as (
    select
        pickup_zip as zona,
        max(fare_amount) as tarifa_maxima,
        min(fare_amount) as tarifa_minima,
        avg(fare_amount) as tarifa_promedio
        from viajes_validos
        group by pickup_zip
)

select
    row_number() over (order by tarifa_promedio desc) as ranking,
    zona,
    tarifa_promedio,
    tarifa_minima,
    tarifa_maxima
from promedios_por_zona
order by tarifa_promedio desc
limit 10;
    

